# Semantic Search Demo

In this notebook, we'll demonstrate semantic search using MiniLM and RAG (Retrieval-Augmented Generation).

In [ ]:
import pandas as pd
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModel
from sklearn.metrics.pairwise import cosine_similarity
import faiss

# Load preprocessed data
train_df = pd.read_csv('data/train_preprocessed.csv')

# Initialize MiniLM model and tokenizer
model_name = 'sentence-transformers/all-MiniLM-L6-v2'
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

# Function to get embeddings
def get_embeddings(texts, batch_size=32):
    embeddings = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        inputs = tokenizer(batch, padding=True, truncation=True, return_tensors='pt', max_length=512)
        with torch.no_grad():
            outputs = model(**inputs)
            batch_embeddings = outputs.last_hidden_state.mean(dim=1)
            embeddings.append(batch_embeddings)
    return torch.cat(embeddings, dim=0)

# Get embeddings for all documents
document_embeddings = get_embeddings(train_df['processed_text'].tolist())

# Create FAISS index
dimension = document_embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(document_embeddings.numpy())

# Function for semantic search
def semantic_search(query, k=5):
    # Get query embedding
    query_embedding = get_embeddings([query])
    
    # Search in FAISS index
    distances, indices = index.search(query_embedding.numpy(), k)
    
    # Return results
    results = []
    for i, idx in enumerate(indices[0]):
        results.append({
            'document': train_df['processed_text'].iloc[idx],
            'label': train_df['label'].iloc[idx],
            'distance': distances[0][i]
        })
    return results

# Demo semantic search
def demo_search(query):
    print(f"\nQuery: {query}")
    print("\nTop 5 most similar documents:")
    results = semantic_search(query)
    for i, result in enumerate(results, 1):
        print(f"\n{i}. Label: {result['label']}")
        print(f"Distance: {result['distance']:.4f}")
        print(f"Text preview: {result['document'][:200]}...")

# Run demo searches
demo_queries = [
    "technology and innovation",
    "financial markets and economy",
    "politics and government"
]

for query in demo_queries:
    demo_search(query)

# Save search results
search_results = {
    'queries': demo_queries,
    'results': {
        query: semantic_search(query) for query in demo_queries
    }
}

import json
with open('data/search_results.json', 'w') as f:
    json.dump(search_results, f)